# Cool Method - Sentiment Arcs (the emotional shape of a story)

An optional starter for **story projects**.

In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset (mount Drive; falls back to a local folder) ---
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
%pip install -q vaderSentiment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _an = SentimentIntensityAnalyzer()
    def sentiment(s): return _an.polarity_scores(s)["compound"]
    print("using VADER sentiment")
except Exception as e:
    # Tiny offline fallback lexicon so the notebook runs in the harness.
    POS=set("happy joy love bright hope warm win light calm safe gentle".split())
    NEG=set("sad fear dark death cold loss cry pain fight alone broken".split())
    def sentiment(s):
        w=s.lower().split(); 
        return (sum(t in POS for t in w)-sum(t in NEG for t in w))/(len(w) or 1)
    print("using offline fallback lexicon:", type(e).__name__)

In [ ]:
import os
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
print("SMOKE mode:", SMOKE)

## Your story, sliced into segments

In class this is your narrative.

In [ ]:
story = ("A bright warm morning, full of hope and love. " * 2 +
         "Then a cold fear, loss and pain, the dark closing in. " * 2 +
         "Alone and broken, a long fight through the night. " * 2 +
         "At last a gentle calm, safe and light, joy returning. " * 2)
words = story.split()
N_SEG = 12
segs = [" ".join(words[i:i+len(words)//N_SEG]) for i in range(0, len(words), max(1,len(words)//N_SEG))][:N_SEG]
raw = np.array([sentiment(s) for s in segs])
print("segment scores:", np.round(raw, 2))

## Smooth it - and watch the window change the shape

A rolling average reveals the arc by averaging out segment-to-segment noise.

*(In Colab you can make this a live `ipywidgets` slider; here we compare two fixed windows so the
notebook is reproducible.)*

In [ ]:
def smooth(x, w):
    if w<=1: return x
    k=np.ones(w)/w
    return np.convolve(x, k, mode="same")

plt.figure(figsize=(7,4))
plt.plot(raw, ".", color="#999", label="raw segments")
plt.plot(smooth(raw,3), label="window=3 (light)")
plt.plot(smooth(raw,7), label="window=7 (heavy)")
plt.axhline(0, color="#ccc", lw=0.8)
plt.title("Emotional arc - the smoothing window changes the shape")
plt.xlabel("story segment"); plt.ylabel("sentiment"); plt.legend(); plt.tight_layout(); plt.show()

## You made an emotional arc

Change `N_SEG` and the smoothing windows and watch the arc breathe.

**Sketch:** plot your own story's arc at two smoothing windows; say where the shape is real and
where the smoothing might be inventing it.